In [3]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

def convert_to_galactic_with_precision(coord, arcsec_precision=0.01):
    """
    Convert SkyCoord (ICRS) to Galactic coordinates, with precision matching input uncertainty.

    Parameters:
    - coord: SkyCoord object in ICRS frame
    - arcsec_precision: uncertainty in arcseconds (default 0.01")

    Returns:
    - (l, b): tuple of galactic longitude and latitude in degrees, rounded to match input precision
    """

    # Convert precision from arcseconds to degrees
    deg_precision = arcsec_precision / 3600.0

    # Determine required decimal places
    decimal_places = max(0, int(abs(np.floor(np.log10(deg_precision)))))
    
    # Convert to galactic coordinates
    gal = coord.galactic
    l_rounded = round(gal.l.deg, decimal_places)
    b_rounded = round(gal.b.deg, decimal_places)

    return l_rounded, b_rounded

# Example usage:
coord = SkyCoord("19h49m29.21s", "-25d12m49.40s", frame='icrs')
l, b = convert_to_galactic_with_precision(coord, arcsec_precision=0.01)

print(f"Galactic coordinates: l = {l}°, b = {b}°")


Galactic coordinates: l = 15.455906°, b = -23.329496°


In [4]:
# Define RA and Dec
coord = SkyCoord("19h49m29.21s", "-25d12m49.40s", frame='icrs')

# Convert to galactic
galactic = coord.galactic

print(f"l = {galactic.l.deg:.6f} deg")
print(f"b = {galactic.b.deg:.6f} deg")

l = 15.455906 deg
b = -23.329496 deg


In [13]:
# Define RA and Dec
coord = SkyCoord("21h27m39.835s", "04d19m45.668s", frame='icrs')

# Convert to galactic
galactic = coord.galactic

print(f"l = {galactic.l.deg:.6f} deg")
print(f"b = {galactic.b.deg:.6f} deg")

l = 57.480995 deg
b = -31.651233 deg


In [5]:
def convert_to_galactic_with_uncertainty(coord, arcsec_uncertainty=0.9, n_samples=10000):
    """
    Convert ICRS SkyCoord to Galactic coordinates with propagated positional uncertainty.

    Parameters:
    - coord: SkyCoord object (ICRS)
    - arcsec_uncertainty: 1-sigma positional error in arcseconds (assumed circular)
    - n_samples: number of Monte Carlo samples (default: 10,000)

    Returns:
    - (l_mean, l_std, b_mean, b_std): Galactic longitude and latitude with 1-sigma uncertainties
    """

    # Convert uncertainty radius to degrees
    err_deg = arcsec_uncertainty / 3600.0

    # Randomly sample offsets within a 2D circular Gaussian
    sigma = err_deg
    dx = np.random.normal(0, sigma, n_samples)
    dy = np.random.normal(0, sigma, n_samples)

    # Adjust RA offset for declination
    dx /= np.cos(coord.dec.radian)

    # Create perturbed coordinates
    perturbed = SkyCoord(
        ra=coord.ra.deg + dx,
        dec=coord.dec.deg + dy,
        unit="deg",
        frame="icrs"
    )

    # Convert to galactic
    gal = perturbed.galactic
    l_vals = gal.l.deg
    b_vals = gal.b.deg

    # Central (nominal) values
    l0 = coord.galactic.l.deg
    b0 = coord.galactic.b.deg

    # Return mean and 1-sigma uncertainty
    return l0, np.std(l_vals), b0, np.std(b_vals)

In [11]:
coord = SkyCoord("19h49m29.21s", "-25d12m49.40s", frame='icrs')
l, l_err, b, b_err = convert_to_galactic_with_uncertainty(coord, arcsec_uncertainty=0.9)

print(f"l = {l:.6f} ± {l_err:.6f} deg")
print(f"b = {b:.6f} ± {b_err:.6f} deg")

l = 15.455906 ± 0.000270 deg
b = -23.329496 ± 0.000249 deg
